# Data Generation

This notebook generates or validates training datasets for multiple geometries in one sequential call without silent try-except wrappers on failure.

Outputs are written under clean_simulation/outputs/data.

In [ ]:
# %% Grid-locked reconstruction setup

from pathlib import Path
import importlib
import json
import os
import sys
import time

import h5py
import numpy as np
import pyvista as pv
import torch
import trimesh
from skimage.measure import marching_cubes

root = Path.cwd().resolve()
while root != root.parent and not (root / "clean_simulation").is_dir():
    root = root.parent
if not (root / "clean_simulation").is_dir():
    raise FileNotFoundError("clean_simulation directory not found from the current working directory")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from clean_simulation.internal_fem_kernel.constants import Material
from clean_simulation.internal_fem_kernel.geometry import HexGrid
from clean_simulation.internal_fem_kernel.queso_integration import _load_queso_dynamically
from clean_simulation.internal_fem_kernel.volume_factory import VolumeFactory
from clean_simulation.internal_nn_runtime import cutcell_bridge

cutcell_bridge = importlib.reload(cutcell_bridge)
prepare_nn_inputs = cutcell_bridge.prepare_nn_inputs
write_geometry_cutcell_training_h5 = cutcell_bridge.write_geometry_cutcell_training_h5

RESOLUTION = (100, 100, 100)
SDF_SUBCELLS = 1
INTEGRATION_ORDER = 2
POLY_ORDER = 1
POISSON_RATIO = 0.3
DEVICE = "cpu"
SDF_BATCH_SIZE = 128
MC_SDF_BATCH_SIZE = 2048
QUESO_PYTHON_PATH = Path(os.environ.get("CUT_CELL_QUESO_PYTHON_PATH", root / "QuESo" / "out" / "install" / "win")).expanduser()


def status(message: str, started_at: float | None = None) -> None:
    elapsed = "" if started_at is None else f" [{time.perf_counter() - started_at:.1f}s]"
    print(f"{message}{elapsed}", flush=True)


def source_grid(volume):
    mins, maxs = np.asarray(volume.mesh.bounds, dtype=np.float64)
    padding = np.maximum((maxs - mins) * 0.10, 1.0e-3)
    raw_lo, raw_hi = mins - padding, maxs + padding
    h = float(np.max(raw_hi - raw_lo) / RESOLUTION[0])
    center = (raw_lo + raw_hi) / 2.0
    half_span = h * np.asarray(RESOLUTION, dtype=np.float64) / 2.0
    return HexGrid(bounds=(tuple(center - half_span), tuple(center + half_span)), resolution=RESOLUTION, device=DEVICE)


def run_queso(mesh_path: Path, grid):
    if not QUESO_PYTHON_PATH.is_dir():
        raise FileNotFoundError("Set CUT_CELL_QUESO_PYTHON_PATH to the QuESo Python install directory.")
    _load_queso_dynamically(str(QUESO_PYTHON_PATH))
    from QuESoPythonModule.PyQuESo import PyQuESo

    settings_path = mesh_path.with_suffix(".queso.json")
    settings_path.write_text(json.dumps({"general_settings": {"input_filename": str(mesh_path), "echo_level": 0, "write_output_to_file": False}, "background_grid_settings": {"grid_type": "b_spline_grid", "lower_bound_xyz": [float(value) for value in grid.bounds[0]], "upper_bound_xyz": [float(value) for value in grid.bounds[1]], "lower_bound_uvw": [0.0, 0.0, 0.0], "upper_bound_uvw": [1.0, 1.0, 1.0], "polynomial_order": [2, 2, 2], "number_of_elements": [int(value) for value in grid.resolution]}, "trimmed_quadrature_rule_settings": {"moment_fitting_residual": 1.0e-8}, "non_trimmed_quadrature_rule_settings": {"integration_method": "Gauss"}}, indent=2), encoding="utf-8")
    try:
        queso = PyQuESo(str(settings_path))
        queso.Run()
        elements = [element for element in queso.GetElements() if element.IsTrimmed()]
    finally:
        settings_path.unlink(missing_ok=True)
    lower = np.asarray(grid.bounds[0], dtype=np.float64)
    spacing = np.array([grid.dx, grid.dy, grid.dz], dtype=np.float64)
    shape = np.asarray(grid.resolution, dtype=np.int64)
    def flat_index(element) -> int:
        center = (np.asarray(element.LowerBoundXYZ()) + np.asarray(element.UpperBoundXYZ())) * 0.5
        ijk = np.rint((center - lower - spacing * 0.5) / spacing).astype(np.int64)
        if np.any(ijk < 0) or np.any(ijk >= shape):
            raise RuntimeError(f"QuESo produced an element outside the locked grid: {center}")
        return int(ijk[0] * shape[1] * shape[2] + ijk[1] * shape[2] + ijk[2])
    pairs = sorted((flat_index(element), element) for element in elements)
    return np.array([flat for flat, _ in pairs], dtype=np.int64), [element for _, element in pairs]


def nodal_sdf(volume, grid, flat_indices: np.ndarray) -> np.ndarray:
    values, _, _ = prepare_nn_inputs(grid=grid, volume=volume, partial_indices=torch.as_tensor(flat_indices, dtype=torch.int64, device=DEVICE), material=Material(E=1.0, nu=POISSON_RATIO), poly_order=POLY_ORDER, refined_subcells=SDF_SUBCELLS, sdf_batch_size=SDF_BATCH_SIZE, sdf_device=DEVICE)
    return values.detach().cpu().numpy()


def _component_polydata(component):
    faces = np.column_stack((np.full(len(component.faces), 3, dtype=np.int64), component.faces)).ravel()
    return pv.PolyData(component.vertices, faces)


def _contains_point(component, point) -> bool:
    probe = pv.PolyData(np.asarray(point, dtype=np.float64).reshape(1, 3))
    selected = probe.select_enclosed_points(_component_polydata(component), tolerance=1.0e-8, check_surface=False)
    return bool(selected["SelectedPoints"][0])


def orient_nested_shells(mesh):
    """Orient each shell from containment parity, preserving enclosed cavities."""
    components = list(mesh.split(only_watertight=False))
    if not components:
        raise RuntimeError("Marching-cubes reconstruction has no connected components")
    for component in components:
        if not component.is_watertight:
            raise RuntimeError("Marching-cubes reconstruction contains a non-watertight component")
        trimesh.repair.fix_normals(component, multibody=False)
    nesting_depths = []
    for component_index, component in enumerate(components):
        nesting_depth = sum(_contains_point(other, component.center_mass) for other_index, other in enumerate(components) if other_index != component_index)
        nesting_depths.append(nesting_depth)
        if nesting_depth % 2:
            component.invert()
    oriented = trimesh.util.concatenate(components)
    signed_volumes = np.asarray([component.volume for component in components], dtype=np.float64)
    expected_signs = np.where(np.asarray(nesting_depths) % 2, -1.0, 1.0)
    if not np.all(np.sign(signed_volumes) == expected_signs):
        raise RuntimeError("Nested-shell orientation does not match containment parity")
    return oriented, {"component_count": len(components), "nesting_depths": nesting_depths, "signed_volumes": signed_volumes.tolist()}


def reconstructed_mesh(source_volume, source_path: Path, grid):
    counts = np.asarray(grid.resolution, dtype=np.int64) * SDF_SUBCELLS + 1
    lower = np.asarray(grid.bounds[0], dtype=np.float64)
    spacing = (np.asarray(grid.bounds[1], dtype=np.float64) - lower) / (counts - 1)
    total_points = int(np.prod(counts))
    yz_count = int(counts[1] * counts[2])
    sdf = np.empty(total_points, dtype=np.float64)
    surface = pv.read(str(source_path))
    for start in range(0, total_points, MC_SDF_BATCH_SIZE):
        stop = min(start + MC_SDF_BATCH_SIZE, total_points)
        flat = np.arange(start, stop, dtype=np.int64)
        i = flat // yz_count
        remainder = flat % yz_count
        j = remainder // counts[2]
        k = remainder % counts[2]
        points = lower + np.column_stack((i, j, k)) * spacing
        sdf[start:stop] = -pv.PolyData(points).compute_implicit_distance(surface)["implicit_distance"]
    sdf = sdf.reshape(tuple(counts))
    if not np.isfinite(sdf).all() or not (sdf.min() < 0.0 < sdf.max()):
        raise RuntimeError("The locked SDF lattice does not enclose a finite zero level set")
    vertices, faces, _, _ = marching_cubes(sdf, level=0.0, spacing=tuple(spacing), allow_degenerate=False)
    mesh = trimesh.Trimesh(vertices=vertices + lower, faces=faces, process=False)
    if len(mesh.faces) == 0:
        raise RuntimeError("Marching-cubes reconstruction has no faces")
    mesh, reconstruction_audit = orient_nested_shells(mesh)
    mesh.metadata["reconstruction_audit"] = reconstruction_audit
    return mesh


def cut_volume_fractions(elements, grid) -> np.ndarray:
    domain_volume = float(np.prod(np.asarray(grid.bounds[1]) - np.asarray(grid.bounds[0])))
    cell_volume = float(grid.dx * grid.dy * grid.dz)
    fractions = [sum(point.Weight() for point in element.GetIntegrationPoints()) * domain_volume / cell_volume for element in elements]
    return np.clip(np.asarray(fractions, dtype=np.float64), 0.0, 1.0)

In [ ]:
# %% Rebuild every target from the parity-oriented reconstruction and preserve matching inputs
from pathlib import Path
import json
import time

import h5py
import numpy as np


root = Path.cwd().resolve()
while root != root.parent and not (root / "clean_simulation").is_dir():
    root = root.parent
if not (root / "clean_simulation").is_dir():
    raise FileNotFoundError("clean_simulation directory not found from the current working directory")

GEOMETRIES = (
    "V7_text", "lvl0_binary", "hanging_sphere_surface", "eggcarton_sphere",
    "thin_ring_sphere", "star_lobe_sphere", "spike_ball",
    *(f"crystal_{index:04d}" for index in range(30)),
)
output_dir = root / "clean_simulation" / "outputs" / "data"
mesh_dir = output_dir / "grid_locked_marching_cubes_stl_material_parity_features"
checkpoint_dir = output_dir / "grid_locked_hybrid_parity_queso_checkpoints"
output_h5 = output_dir / "grid_locked_source_magnitude_material_parity_rebuilt_queso_i2_p1_simple_harsh_crystals30.h5"
mesh_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

missing_stls = [
    name for name in GEOMETRIES
    if not (root / "clean_simulation" / "geometries" / f"{name}.stl").is_file()
]
if missing_stls:
    raise FileNotFoundError(f"Missing source STLs: {missing_stls}")

orientation_by_geometry = {}
reconstruction_audit_by_geometry = {}
run_started_at = time.perf_counter()
for geometry_number, geometry_name in enumerate(GEOMETRIES, start=1):
    geometry_started_at = time.perf_counter()
    label = f"[{geometry_number:02d}/{len(GEOMETRIES):02d}] {geometry_name}"
    source_path = root / "clean_simulation" / "geometries" / f"{geometry_name}.stl"
    source_volume = VolumeFactory.from_file(source_path)
    source_volume.device = DEVICE
    grid = source_grid(source_volume)

    rebuilt_mesh = reconstructed_mesh(source_volume, source_path, grid)
    reconstruction_audit = rebuilt_mesh.metadata["reconstruction_audit"]
    reconstruction_audit_by_geometry[geometry_name] = reconstruction_audit
    rebuilt_path = mesh_dir / f"{geometry_name}_gridlocked_sdf{SDF_SUBCELLS}.stl"
    rebuilt_mesh.export(rebuilt_path)
    rebuilt_volume = VolumeFactory.from_file(rebuilt_path)
    rebuilt_volume.device = DEVICE

    rebuilt_indices, rebuilt_elements = run_queso(rebuilt_path, grid)
    if rebuilt_indices.size == 0:
        raise RuntimeError(f"{geometry_name}: QuESo found no rebuilt-geometry trimmed cells")
    if not np.all(rebuilt_indices[1:] > rebuilt_indices[:-1]):
        raise RuntimeError(f"{geometry_name}: QuESo flat indices are not strictly sorted")

    geometry_h5 = checkpoint_dir / f"{geometry_name}.h5"
    result = write_geometry_cutcell_training_h5(
        output_h5=geometry_h5,
        group_name="samples",
        grid=grid,
        volume=rebuilt_volume,
        material=Material(E=1.0, nu=POISSON_RATIO),
        poly_order=POLY_ORDER,
        order=INTEGRATION_ORDER,
        device=DEVICE,
        batch_size=SDF_BATCH_SIZE,
        replace_existing=True,
        refined_subcells=SDF_SUBCELLS,
        queso_elements=rebuilt_elements,
        include_sdf=False,
    )
    if result["num_cells"] != len(rebuilt_indices):
        raise RuntimeError(f"{geometry_name}: writer rows do not match rebuilt QuESo cells")

    rebuilt_fractions = cut_volume_fractions(rebuilt_elements, grid)
    with h5py.File(geometry_h5, "a") as geometry_file:
        group = geometry_file["samples"]
        flat_index = group["flat_index"][...]
        fraction_positions = np.searchsorted(rebuilt_indices, flat_index)
        if (
            np.any(fraction_positions >= len(rebuilt_indices))
            or not np.array_equal(rebuilt_indices[fraction_positions], flat_index)
        ):
            raise RuntimeError(f"{geometry_name}: writer and rebuilt QuESo selected different cells")

        source_d_values = nodal_sdf(source_volume, grid, flat_index)
        material_d_values = nodal_sdf(rebuilt_volume, grid, flat_index)
        volume_fraction = rebuilt_fractions[fraction_positions]
        orientation_correlation = float(np.corrcoef(material_d_values.mean(axis=1) - 0.5, volume_fraction)[0, 1])
        if not np.isfinite(orientation_correlation) or abs(orientation_correlation) < 0.9:
            raise RuntimeError(
                f"{geometry_name}: ambiguous reconstructed-SDF/QuESo material orientation "
                f"(correlation={orientation_correlation:.3f})"
            )
        material_sdf_was_complemented = orientation_correlation < 0.0
        if material_sdf_was_complemented:
            material_d_values = 1.0 - material_d_values

        # Preserve the accurate source-SDF magnitude, but impose the parity-aware
        # reconstructed mesh material side that QuESo integrates.
        d_values = 0.5 + np.where(material_d_values >= 0.5, 1.0, -1.0) * np.abs(source_d_values - 0.5)
        orientation_by_geometry[geometry_name] = {
            "correlation_before_alignment": orientation_correlation,
            "material_sdf_was_complemented": material_sdf_was_complemented,
        }

        if not np.isfinite(d_values).all() or not np.isfinite(group["stiffness_ut"][...]).all():
            raise RuntimeError(f"{geometry_name}: non-finite hybrid SDF or stiffness target")
        group.create_dataset("d_values", data=d_values)
        group.create_dataset("cut_volume_fraction", data=volume_fraction)
        group.create_dataset("geometry_name", data=np.full(len(flat_index), geometry_name.encode("utf-8"), dtype="S64"))
        group.attrs["material_sdf_volume_correlation_before_alignment"] = orientation_correlation
        group.attrs["material_sdf_was_complemented"] = material_sdf_was_complemented
        group.attrs["feature_representation"] = "source_sdf_magnitude_with_reconstructed_mesh_material_parity"
        group.attrs["reconstruction_audit"] = json.dumps(reconstruction_audit)

    status(
        f"{label}: retained all {len(rebuilt_indices):,} rebuilt QuESo cut cells; "
        f"reconstructed-SDF/volume correlation={orientation_correlation:+.3f}; "
        f"shells={reconstruction_audit['component_count']}, depths={reconstruction_audit['nesting_depths']}",
        geometry_started_at,
    )

combined_chunks = {"flat_index": [], "d_values": [], "stiffness_ut": [], "cut_volume_fraction": [], "geometry_name": []}
for geometry_name in GEOMETRIES:
    with h5py.File(checkpoint_dir / f"{geometry_name}.h5", "r") as geometry_file:
        group = geometry_file["samples"]
        for name in combined_chunks:
            combined_chunks[name].append(group[name][...])
combined = {name: np.concatenate(chunks, axis=0) for name, chunks in combined_chunks.items()}

with h5py.File(output_h5, "w") as output_file:
    group = output_file.create_group("integration_2_poly_1")
    for name, values in combined.items():
        group.create_dataset(name, data=values)
    group.attrs["input_source"] = "source_sdf_magnitude_with_parity_oriented_reconstructed_mesh_material_sign"
    group.attrs["target_source"] = "same_parity_oriented_rebuilt_grid_locked_marching_cubes_queso_mesh"
    group.attrs["training_representation"] = "raw_physical_space"
    group.attrs["cell_selection"] = "all_rebuilt_queso_trimmed_cells"
    group.attrs["selection_filter"] = "none"
    group.attrs["source_geometries"] = json.dumps(GEOMETRIES)
    group.attrs["crystal_count"] = 30
    group.attrs["sdf_subcells"] = SDF_SUBCELLS
    group.attrs["stiffness_source"] = "rebuilt_grid_locked_marching_cubes_queso"
    group.attrs["reconstruction"] = "global_signed_distance_marching_cubes_on_locked_source_grid_with_containment_parity"
    group.attrs["sdf_orientation_alignment"] = json.dumps(orientation_by_geometry)
    group.attrs["reconstruction_audit"] = json.dumps(reconstruction_audit_by_geometry)

status(
    f"Done: {len(combined['flat_index']):,} source-magnitude/material-parity / rebuilt-QuESo rows written to {output_h5.name}",
    run_started_at,
)

Found 8418 cut cells via QuESo.
Done. Wrote 8418 cut-cell entries to C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\data\grid_locked_source_sdf_rebuilt_queso_checkpoints\V7_text.h5:samples
[01/37] V7_text: retained all 8,418 rebuilt QuESo cut cells [108.5s]
Found 6482 cut cells via QuESo.
Done. Wrote 6482 cut-cell entries to C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\data\grid_locked_source_sdf_rebuilt_queso_checkpoints\lvl0_binary.h5:samples
[02/37] lvl0_binary: retained all 6,482 rebuilt QuESo cut cells [107.0s]
Found 30608 cut cells via QuESo.
Done. Wrote 30608 cut-cell entries to C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\data\grid_locked_source_sdf_rebuilt_queso_checkpoints\hanging_sphere_surface.h5:samples
[03/37] hanging_sphere_surface: retained all 30,608 rebuilt QuESo cut cells [51.5s]
Found 66240 cut cells via QuESo.
Done. Wrote 66240 cut-cell entries to C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\data\grid_locked_source_

In [ ]:
# %% Post-generation cleaning: remove per-cell inside/outside orientation outliers.
from pathlib import Path

from clean_simulation.scripts.data.orientation_filter import filter_orientation_outliers

LOW_CORNER_THRESHOLD = 0.25
HIGH_CVF_THRESHOLD = 0.75

root = Path.cwd().resolve()
while root != root.parent and not (root / "clean_simulation").is_dir():
    root = root.parent
if not (root / "clean_simulation").is_dir():
    raise FileNotFoundError("clean_simulation directory not found from the current working directory")

output_h5 = root / "clean_simulation" / "outputs" / "data" / "grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30.h5"
result = filter_orientation_outliers(
    output_h5,
    low_corner_threshold=LOW_CORNER_THRESHOLD,
    high_volume_threshold=HIGH_CVF_THRESHOLD,
)

print(f"Total rows: {result['total_rows']:,}")
print(f"  removed -- non-finite d/cut_volume_fraction: {result['removed_nonfinite']:,}")
print(f"  removed -- per-cell orientation outliers:     {result['removed_outliers']:,}")
print(f"  retained:                                     {result['retained_rows']:,}")
print()
print(f"{'geometry':30s} {'n_removed':>10s}")
print("-" * 42)
for name, count in result["removed_by_geometry"].items():
    print(f"{name:30s} {count:10,d}")
print(f"\nWrote cleaned dataset to {result['output_h5'].name}")

Total rows: 1,151,274
  removed -- non-finite d/cut_volume_fraction: 0
  removed -- per-cell orientation outliers:     1,770
  retained:                                     1,149,504

geometry                        n_removed
------------------------------------------
crystal_0000                           21
crystal_0001                            1
crystal_0002                          190
crystal_0003                            6
crystal_0004                            9
crystal_0006                           11
crystal_0007                            1
crystal_0008                            3
crystal_0010                          118
crystal_0011                            8
crystal_0012                            1
crystal_0013                          129
crystal_0014                          105
crystal_0015                           19
crystal_0016                            2
crystal_0017                          224
crystal_0018                            7
crystal_0019     